# Retrieval-Augmented Generation (RAG) System
This notebook demonstrates a complete RAG pipeline.

## Section 1: Setup and Installations


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -U langchain unstructured langchain-community langchain-text-splitters langchain-chroma langchain-groq

In [3]:
# Import necessary modules for RAG and set environment variables
import os
#from langchain.document_loaders import UnstructuredFileLoader #old import not working
from langchain_community.document_loaders import UnstructuredFileLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA

In [4]:
# Configure Groq API Key
os.environ["GROQ_API_KEY"] = "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"

## Section 2: Data Acquisition


In [5]:
import requests
url = "https://www.mathcentre.ac.uk/resources/uploaded/mc-ty-introfns-2009-1.pdf"
response = requests.get(url)

In [6]:
with open("python_introduction.pdf", "wb") as f:
  f.write(response.content)

In [7]:
!pip install unstructured

## Section 3: Document Loading


In [8]:
loader = UnstructuredFileLoader("python_introduction.pdf")

/tmp/ipykernel_6738/1921939299.py:1: LangChainDeprecationWarning: The class `UnstructuredFileLoader` was deprecated in LangChain 0.2.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-unstructured package and should be used instead. To use it run `pip install -U `langchain-unstructured` and import as `from `langchain_unstructured import UnstructuredLoader``.
  loader = UnstructuredFileLoader("python_introduction.pdf")


In [9]:
!pip install "unstructured[pdf]"

In [10]:
documents = loader.load()
documents

[Document(metadata={'source': 'python_introduction.pdf'}, page_content='Introduction to functions\n\nmc-TY-introfns-2009-1\n\nA function is a rule which operates on one number to give another number. However, not every rule describes a valid function. This unit explains how to see whether a given rule describes a valid function, and introduces some of the mathematical terms associated with functions.\n\nIn order to master the techniques explained here it is vital that you undertake plenty of practice exercises so that they become second nature.\n\nAfter reading this text, and/or viewing the video tutorial on this topic, you should be able to:\n\nrecognise when a rule describes a valid function,\n\nbe able to plot the graph of a part of a function,\n\nﬁnd a suitable domain for a function, and ﬁnd the corresponding range.\n\nContents\n\n1. What is a function? 2. Plotting the graph of a function\n\n2\n\n3\n\n3. When is a function valid?\n\n4\n\n4. Some further examples\n\n6\n\nwww.mathcen

## Section 4: Text Splitting


In [11]:
text_splitter = CharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=400
)

In [12]:
texts = text_splitter.split_documents(documents)

In [14]:
type(texts)

list

In [15]:
len(texts)

10

In [16]:
texts[4]

Document(metadata={'source': 'python_introduction.pdf'}, page_content='0.\n\n≥\n\n0,\n\nf(x)\n\n0,\n\n≥ 0, and the range is the corresponding\n\n≥\n\nKey Point\n\nThe domain of a function is the set of possible inputs. The range of a function is the set of corresponding outputs.\n\nwww.mathcentre.ac.uk\n\n5\n\nc(cid:13) mathcentre 2009\n\n4. Some further examples\n\nExample\n\nConsider the function\n\nf(x) = 2x2\n\n−\n\n3x + 5.\n\nTo make sure that the function is valid, we need to check whether we get exactly one output for each input, and whether there needs to be any restriction on the domain. As before, we can calculate the output of this function at some speciﬁc values to help us with plotting our graph:\n\nf(0) = 2 = 5, f(1) = 2 = 2 = 4, f(2) = 2 = 8 = 7, f(3) = 2 = 18 = 14,\n\n×\n\n02\n\n−\n\n3\n\n× −\n\n12 3 − 3 + 5\n\n× −\n\n22 3 − 6 + 5\n\n32 3 − 9 + 5\n\n× −\n\n×\n\n×\n\n×\n\n×\n\n0 + 5\n\n1 + 5\n\n2 + 5\n\n3 + 5\n\nf(\n\n−\n\n1)2 × = 2 + 3 + 5 = 10.\n\n1) = 2\n\n(\n\n−\n\n−

## Section 5: Vector Store & Embeddings


In [17]:
embeddings = HuggingFaceEmbeddings()

/tmp/ipykernel_6738/3655315981.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/tmp/ipykernel_6738/3655315981.py:1: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/to

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
persist_directory = "vector_db"

In [19]:
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=persist_directory
)

In [20]:
retriever = vectordb.as_retriever()

In [23]:
# llm from groq
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

## Section 6: RAG Chain Setup


In [24]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

In [29]:
query = "What is function?"
response = qa_chain.invoke({"query": query})

In [30]:
print(response)

{'query': 'What is function?', 'result': 'A function is a rule that maps a number to another unique number. In other words, it is a rule that takes an input (called the argument or independent variable) and gives a corresponding output (called the dependent variable). For a rule to be considered a valid function, it must give exactly one output for each input.', 'source_documents': [Document(id='117239a2-4145-4040-92c9-d91cf3d1744b', metadata={'source': 'python_introduction.pdf'}, page_content='Introduction to functions\n\nmc-TY-introfns-2009-1\n\nA function is a rule which operates on one number to give another number. However, not every rule describes a valid function. This unit explains how to see whether a given rule describes a valid function, and introduces some of the mathematical terms associated with functions.\n\nIn order to master the techniques explained here it is vital that you undertake plenty of practice exercises so that they become second nature.\n\nAfter reading this

In [31]:
print(response["result"])

A function is a rule that maps a number to another unique number. In other words, it is a rule that takes an input (called the argument or independent variable) and gives a corresponding output (called the dependent variable). For a rule to be considered a valid function, it must give exactly one output for each input.


In [32]:
# invoke the qa chain and get a response for user query
query = "Give me summary of all function from this pdf?"
response = qa_chain.invoke({"query": query})
print(response["result"])
print("*"*30)
print("Source Document:", response["source_documents"][0].metadata["source"])

The PDF discusses several functions, including:

1. **f(x) = x + 3**: a simple function that adds 3 to any input.
2. **f(x) = 2x^2 + 5x - 3**: a quadratic function with a table of values and a graph.
3. **f(x) = 1/x**: a function that takes the reciprocal of the input, with a discussion of its domain and range.
4. **f(x) = (x - 1)^2**: a quadratic function with a graph and a discussion of its domain and range.
5. **f(x) = 3√x**: a function that takes the cube root of the input, with a discussion of its domain and range.
6. **f(x) = 2sin(x)**: a trigonometric function with a mention of its domain and range.
7. **f(x) = x^2 - 6x + 9**: a quadratic function with a mention of its domain and range.
8. **f(x) = 2e^(-x)**: an exponential function with a mention of its domain and range.
9. **f(x) = 4 - x^2**: a quadratic function with a mention of its domain and range.
10. **f(x) = 2x - 1**: a linear function with a mention of its domain and range.
11. **f(x) = 3e^(5x)**: an exponential functi

## Section 7: Interactive Interface (Gradio)


In [33]:
pip install gradio

In [34]:
import gradio as gr

def ask_question(query):
    response = qa_chain.invoke({"query": query})
    return response["result"]

In [35]:
iface = gr.Interface(
    fn=ask_question,
    inputs=gr.Textbox(lines=2, placeholder="Enter your query here..."),
    outputs="text",
    title="RAG System Query Interface",
    description="Ask questions about the PDF document."
)

iface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5eb98dab58ffafbc49.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://5eb98dab58ffafbc49.gradio.live
